In [2]:
%pip install numpy scipy tifffile scikit-image
# optional: imageio, ome-types if you want richer OME metadata handling

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# normalize_single_channel_files_notebook.py
# Paste into a Jupyter cell.

import warnings
from pathlib import Path
import numpy as np
from tifffile import imread, imwrite
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
import pandas as pd
from typing import Dict, Any, List

# filenames to skip (exact, case-sensitive)
EXCLUDED_FILENAMES = {
    "dapi.tif",
    "18s_RNA.tif",
    "AlphaSMA_Vimentin.tif",
    "ATP_Cadherin_CD45.tif",
    "regions__Subregion.tif",
    "segmentation_labels.tiff",  # double 'f' intentional
}

# --- leftmost mode estimator (same robust strategy as before) ---
def find_leftmost_mode(
    data: np.ndarray,
    kde_bandwidth=None,
    eval_grid_points: int = 2048,
    hist_bins: int = 256,
    min_prominence: float = 1e-6,
    ignore_zeros: bool = False,
) -> float:
    """
    Estimate the leftmost histogram mode of 1D data.
    """
    data = np.asarray(data)
    data = data[np.isfinite(data)]
    if ignore_zeros:
        data = data[data != 0]
    if data.size == 0:
        raise ValueError("Empty data after filtering in find_leftmost_mode()")

    # discrete shortcut for small integer ranges
    if np.issubdtype(data.dtype, np.integer) and data.size > 1000 and data.max() - data.min() < 65536:
        try:
            offset = int(data.min())
            counts = np.bincount((data.astype(np.int64) - offset).astype(np.int64))
            peak_idx = int(np.argmax(counts))
            return float(offset + peak_idx)
        except Exception:
            pass

    vmin, vmax = float(np.min(data)), float(np.max(data))
    if vmax == vmin:
        return float(vmin)

    grid = np.linspace(vmin, vmax, eval_grid_points)
    try:
        kde = gaussian_kde(data, bw_method=kde_bandwidth)
        density = kde(grid)
        prom = max(min_prominence * np.max(density), 1e-12)
        peaks, _ = find_peaks(density, prominence=prom)
        if peaks.size > 0:
            leftmost_idx = peaks[np.argmin(grid[peaks])]
            return float(grid[leftmost_idx])
    except Exception as e:
        warnings.warn(f"KDE failed ({e}); falling back to histogram method.")

    # histogram fallback
    hist_counts, hist_edges = np.histogram(data, bins=hist_bins)
    max_c = hist_counts.max()
    candidates = np.where(hist_counts >= max(1, 0.5 * max_c))[0]
    if candidates.size > 0:
        leftmost = int(candidates.min())
        return float(0.5 * (hist_edges[leftmost] + hist_edges[leftmost + 1]))

    # last fallback
    return float(np.percentile(data, 1.0))


# --- process a single 2D image (single-channel file) ---
def process_single_channel_file(
    infile: Path,
    shift: float,
    *,
    ignore_zeros: bool = False,
    overwrite: bool = False,
) -> Dict[str, Any]:
    """
    Apply an additive shift to infile image and save.
    - shift: value to add to the pixel intensities (can be negative)
    - overwrite: if True write back to infile, else write infile.stem + "_normalized" + suffix
    Returns dict with processing info.
    """
    result = {"input_path": str(infile), "output_path": None, "success": False, "error": None}
    try:
        arr = imread(str(infile))
        if arr.ndim != 2:
            # accept also single-page (2D). If it's >2D, try to reduce to 2D if it's (1, Y, X) etc.
            if arr.ndim == 3 and 1 in arr.shape:
                # collapse singleton dim
                arr = arr.squeeze()
            else:
                raise ValueError(f"Expected 2D single-channel image; found shape {arr.shape}")

        orig_dtype = arr.dtype

        # compute clipping bounds from dtype
        if np.issubdtype(orig_dtype, np.integer):
            lo, hi = np.iinfo(orig_dtype).min, np.iinfo(orig_dtype).max
        else:
            lo, hi = float(np.nanmin(arr)), float(np.nanmax(arr))

        out_arr = arr.astype(np.float64) + float(shift)
        out_arr = np.clip(out_arr, lo, hi)

        # cast back to original dtype
        out_cast = out_arr.astype(orig_dtype)

        if overwrite:
            out_path = infile
        else:
            out_path = infile.with_name(infile.stem + infile.suffix)

        imwrite(str(out_path), out_cast)
        result.update({"output_path": str(out_path), "success": True})
    except Exception as e:
        result.update({"error": str(e)})
        warnings.warn(f"Failed processing {infile}: {e}")
    return result


# --- high-level: normalize each folder where each file is a single channel ---
def normalize_parent_folder_single_channel_files(
    parent_folder: str | Path,
    *,
    reference: str | float = "median",
    ignore_zeros: bool = False,
    overwrite: bool = False,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Walk each direct subfolder of parent_folder. Each TIFF-like file in the subfolder is
    treated as a single-channel image (one marker). Compute leftmost mode per file,
    compute a reference for the folder (default=median of modes), compute shifts, apply and save.
    Returns pandas DataFrame summary with columns: parent_folder, filename, mode, shift, output_path, processed, error.
    """
    parent = Path(parent_folder)
    if not parent.exists() or not parent.is_dir():
        raise ValueError(f"Parent folder does not exist or is not a directory: {parent}")

    records: List[Dict[str, Any]] = []

    # iterate direct subfolders only
    for sub in sorted([p for p in parent.iterdir() if p.is_dir()]):
        if verbose:
            print(f"\nProcessing folder: {sub.name}")
        # collect candidate image files in this folder
        tiff_files = sorted([p for p in sub.iterdir() if p.is_file() and p.name not in EXCLUDED_FILENAMES and p.suffix.lower() in ('.tif', '.tiff')])
        if verbose:
            print(f" Found {len(tiff_files)} files (excluded list removed).")

        # if no TIFFs to process, add records for excluded or empty
        if len(tiff_files) == 0:
            # record excluded files for completeness
            for p in sorted([q for q in sub.iterdir() if q.is_file() and q.suffix.lower() in ('.tif', '.tiff')]):
                rec = {
                    "parent_folder": sub.name,
                    "filename": p.name,
                    "mode": None,
                    "shift": None,
                    "output_path": None,
                    "processed": False,
                    "error": "no_processable_files_in_folder_or_all_excluded",
                }
                records.append(rec)
            continue

        # compute leftmost mode for each file
        modes = []
        for p in tiff_files:
            try:
                arr = imread(str(p))
                if arr.ndim != 2:
                    # attempt to squeeze singleton dimensions (common if saved as (1,y,x))
                    if arr.ndim == 3 and 1 in arr.shape:
                        arr = arr.squeeze()
                        if arr.ndim != 2:
                            raise ValueError(f"Could not coerce to 2D, shape now {arr.shape}")
                    else:
                        raise ValueError(f"File has unexpected ndim {arr.ndim}, shape {arr.shape}")
                flat = arr.ravel().astype(np.float64)
                mode_val = find_leftmost_mode(flat, ignore_zeros=ignore_zeros)
                modes.append((p, float(mode_val)))
                if verbose:
                    print(f"  {p.name}: mode {mode_val:.4f}")
            except Exception as e:
                warnings.warn(f"Failed to estimate mode for {p}: {e}")
                modes.append((p, None))

        # determine reference value (median/min/zero/numeric)
        valid_modes = [m for (_, m) in modes if m is not None]
        if len(valid_modes) == 0:
            if verbose:
                print("  No valid modes found in this folder; skipping normalization.")
            # record errors and continue
            for p, m in modes:
                records.append({
                    "parent_folder": sub.name,
                    "filename": p.name,
                    "mode": m,
                    "shift": None,
                    "output_path": None,
                    "processed": False,
                    "error": "no_valid_modes_in_folder",
                })
            continue

        if reference == "median":
            ref_val = float(np.median(valid_modes))
        elif reference == "min":
            ref_val = float(np.min(valid_modes))
        elif reference == "zero":
            ref_val = 0.0
        elif isinstance(reference, (int, float)):
            ref_val = float(reference)
        else:
            raise ValueError("reference must be 'median', 'min', 'zero', or a numeric value")

        if verbose:
            print(f" Folder reference value = {ref_val:.4f}")

        # apply shifts and save per file
        for p, mode_val in modes:
            if mode_val is None:
                records.append({
                    "parent_folder": sub.name,
                    "filename": p.name,
                    "mode": None,
                    "shift": None,
                    "output_path": None,
                    "processed": False,
                    "error": "mode_estimation_failed",
                })
                continue
            shift = ref_val - float(mode_val)
            res = process_single_channel_file(p, shift=shift, ignore_zeros=ignore_zeros, overwrite=overwrite)
            records.append({
                "parent_folder": sub.name,
                "filename": p.name,
                "mode": float(mode_val),
                "shift": float(shift),
                "output_path": res.get("output_path"),
                "processed": res.get("success", False),
                "error": res.get("error"),
            })
            if verbose:
                status = "OK" if res.get("success", False) else f"ERR: {res.get('error')}"
                print(f"  {p.name}: shift {shift:.3f} -> {status}")

    df = pd.DataFrame.from_records(records)
    return df


In [3]:

# ---------------------------
# Example usage in notebook:
#
PARENT_FOLDER = "D:\\Dom\\Fibrosis project\\4th year data\\CellTune\\Data for normalisation"
DF = normalize_parent_folder_single_channel_files(
     PARENT_FOLDER,
     reference="median",   # or "min" or "zero" or numeric
     ignore_zeros=True,
     overwrite=False,
     verbose=True
 )
display(DF.head(200))
#DF.to_csv("normalization_summary.csv", index=False)

# ---------------------------



Processing folder: R2r1
 Found 24 files (excluded list removed).
  Caveolin.tif: mode 8.2047
  CD11c.tif: mode 29.9707
  CD20.tif: mode 6.2042
  CD206.tif: mode 645.6629
  CD31.tif: mode 75.8666
  CD36.tif: mode 278.8481
  CD3e.tif: mode 114.6914
  CD4.tif: mode 21.8984
  CD44.tif: mode 191.2433
  CD45.tif: mode 69.1035
  CD45R.tif: mode 4.0630
  CD68.tif: mode 94.3379
  CD8.tif: mode 119.8701
  ColA1.tif: mode 32.9116
  F480.tif: mode 22.9834
  FcYr.tif: mode 33.0713
  FoxP3.tif: mode 227.1886
  Iba1.tif: mode 82.2129
  Ki67.tif: mode 372.2496
  Ly6G.tif: mode 206.9697
  PanCK.tif: mode 234.3004
  S100A9.tif: mode 2.0703
  Ter119.tif: mode 43.9116
  Vimentin.tif: mode 51.8842
 Folder reference value = 72.4851
  Caveolin.tif: shift 64.280 -> OK
  CD11c.tif: shift 42.514 -> OK
  CD20.tif: shift 66.281 -> OK
  CD206.tif: shift -573.178 -> OK
  CD31.tif: shift -3.382 -> OK
  CD36.tif: shift -206.363 -> OK
  CD3e.tif: shift -42.206 -> OK
  CD4.tif: shift 50.587 -> OK
  CD44.tif: shift -11

,parent_folder,filename,mode,shift,output_path,processed,error
0,R2r1,Caveolin.tif,8.204690,64.280385,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
1,R2r1,CD11c.tif,29.970689,42.514386,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
2,R2r1,CD20.tif,6.204201,66.280874,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
3,R2r1,CD206.tif,645.662921,-573.177846,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
4,R2r1,CD31.tif,75.866634,-3.381559,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
...,...,...,...,...,...,...,...
195,R6r2,CD206.tif,11.644358,45.604885,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
196,R6r2,CD31.tif,3.618955,53.630288,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
197,R6r2,CD36.tif,180.972643,-123.723400,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
198,R6r2,CD3e.tif,77.621094,-20.371851,D:\Dom\Fibrosis project\4th year data\CellTune...,True,None
